# 02 MA Enrollment EDA

Purpose: turn the verified CMS MA State/County/Plan-Type files into an analysis-ready table and understand the enrollment trends deeply enough to support forecasting and opportunity analytics.

Design rule for this notebook: **do not drop source rows**. CMS uses `.` in `Enrolled` where values are suppressed or unavailable. Those rows remain in the processed data with explicit flags, while numeric aggregations use the parsed enrollment value when available.

## EDA Plan

What this notebook does and why:

1. Load every monthly MA SCP ZIP so the EDA reflects the full installed dataset, not a sample.
2. Standardize column names and preserve raw values so later bugs can be traced back to CMS source files.
3. Add quality flags instead of dropping rows, because suppression/missingness is itself useful context.
4. Create national, state, county, and plan-type summaries for forecasting and business-intelligence views.
5. Quantify growth, volatility, and concentration so the next notebook can choose sensible forecasting targets.

In [1]:
from __future__ import annotations

import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
MA_SCP_DIR = RAW_DIR / "ma_scp"
PROCESSED_DIR = ROOT / "data" / "processed"
REPORT_TABLE_DIR = ROOT / "reports" / "tables"
FIGURE_DIR = ROOT / "reports" / "figures"

for path in [PROCESSED_DIR, REPORT_TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

EXPECTED_MONTHS = pd.period_range("2024-01", "2026-05", freq="M").astype(str).tolist()

print(f"Project root: {ROOT}")
print(f"MA SCP directory: {MA_SCP_DIR}")

Project root: D:\Project 1\rcm-cms-mvp
MA SCP directory: D:\Project 1\rcm-cms-mvp\data\raw\ma_scp


## Load All MA SCP Files

Reason: forecasting and growth analytics are sensitive to missing months. This step reads every installed monthly file and attaches source metadata (`source_zip`, `source_csv`, `report_month`) to every row for traceability.

In [2]:
def month_from_zip_name(path: Path) -> str:
    match = re.search(r"(20\d{2}-\d{2})", path.name)
    if not match:
        raise ValueError(f"Could not parse YYYY-MM from {path.name}")
    return match.group(1)


def read_ma_scp_zip(zip_path: Path) -> pd.DataFrame:
    report_month = month_from_zip_name(zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        csv_names = [name for name in zf.namelist() if name.lower().endswith(".csv")]
        if len(csv_names) != 1:
            raise ValueError(f"Expected exactly one CSV in {zip_path.name}, found {csv_names}")
        csv_name = csv_names[0]
        with zf.open(csv_name) as handle:
            df = pd.read_csv(handle, dtype=str, keep_default_na=False)

    df["source_zip"] = zip_path.name
    df["source_csv"] = csv_name
    df["report_month"] = report_month
    return df


zip_paths = sorted(MA_SCP_DIR.glob("*.zip"))
print(f"MA SCP ZIP files found: {len(zip_paths)}")

raw_frames = [read_ma_scp_zip(path) for path in zip_paths]
raw_ma = pd.concat(raw_frames, ignore_index=True)
raw_ma["source_row_number"] = raw_ma.groupby("source_zip").cumcount() + 1

display(raw_ma.head())
print(f"Raw rows loaded: {len(raw_ma):,}")
print(f"Columns: {raw_ma.columns.tolist()}")

MA SCP ZIP files found: 29


,County,State,PLAN TYPE,SSA Code,FIPS Code,Enrolled,source_zip,source_csv,report_month,source_row_number
0,Autauga,AL,HCPP - 1833 Cost,01000,01001,.,ma-scp-2024-01.zip,SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv,2024-01,1
1,Autauga,AL,HMO/HMOPOS,01000,01001,3689,ma-scp-2024-01.zip,SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv,2024-01,2
2,Autauga,AL,LI NET Sponsor,01000,01001,.,ma-scp-2024-01.zip,SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv,2024-01,3
3,Autauga,AL,Local PPO,01000,01001,3416,ma-scp-2024-01.zip,SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv,2024-01,4
4,Autauga,AL,MSA,01000,01001,.,ma-scp-2024-01.zip,SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv,2024-01,5


Raw rows loaded: 590,398
Columns: ['County', 'State', 'PLAN TYPE', 'SSA Code', 'FIPS Code', 'Enrolled', 'source_zip', 'source_csv', 'report_month', 'source_row_number']


## Schema And Month Coverage

Reason: before cleaning, verify the source schema is stable. If CMS changes column names in one month, the model pipeline can silently break unless we catch it here.

In [3]:
schema_by_file = []
for path, frame in zip(zip_paths, raw_frames):
    schema_by_file.append(
        {
            "file_name": path.name,
            "report_month": month_from_zip_name(path),
            "row_count": len(frame),
            "columns": tuple(frame.drop(columns=["source_zip", "source_csv", "report_month"]).columns),
        }
    )

schema_df = pd.DataFrame(schema_by_file)
schema_df["schema_id"] = schema_df["columns"].astype(str).factorize()[0] + 1

coverage_df = pd.DataFrame({"report_month": EXPECTED_MONTHS}).merge(
    schema_df[["report_month", "file_name", "row_count", "schema_id"]],
    on="report_month",
    how="left",
)
coverage_df["present"] = coverage_df["file_name"].notna()

schema_df.to_csv(REPORT_TABLE_DIR / "ma_scp_schema_by_file.csv", index=False)
coverage_df.to_csv(REPORT_TABLE_DIR / "ma_scp_coverage_for_eda.csv", index=False)

display(schema_df.head())
display(coverage_df)

assert coverage_df["present"].all(), "Missing one or more expected MA SCP months."
assert schema_df["schema_id"].nunique() == 1, "MA SCP schema varies across files. Review before continuing."
print("PASS: all expected months are present and schema is stable.")

,file_name,report_month,row_count,columns,schema_id
0,ma-scp-2024-01.zip,2024-01,20668,"(County, State, PLAN TYPE, SSA Code, FIPS Code, Enrolled)",1
1,ma-scp-2024-02.zip,2024-02,20602,"(County, State, PLAN TYPE, SSA Code, FIPS Code, Enrolled)",1
2,ma-scp-2024-03.zip,2024-03,20548,"(County, State, PLAN TYPE, SSA Code, FIPS Code, Enrolled)",1
3,ma-scp-2024-04.zip,2024-04,20549,"(County, State, PLAN TYPE, SSA Code, FIPS Code, Enrolled)",1
4,ma-scp-2024-05.zip,2024-05,20514,"(County, State, PLAN TYPE, SSA Code, FIPS Code, Enrolled)",1


,report_month,file_name,row_count,schema_id,present
0,2024-01,ma-scp-2024-01.zip,20668,1,True
1,2024-02,ma-scp-2024-02.zip,20602,1,True
2,2024-03,ma-scp-2024-03.zip,20548,1,True
3,2024-04,ma-scp-2024-04.zip,20549,1,True
4,2024-05,ma-scp-2024-05.zip,20514,1,True
5,2024-06,ma-scp-2024-06.zip,20478,1,True
6,2024-07,ma-scp-2024-07.zip,20482,1,True
7,2024-08,ma-scp-2024-08.zip,20480,1,True
8,2024-09,ma-scp-2024-09.zip,20491,1,True
9,2024-10,ma-scp-2024-10.zip,20443,1,True


PASS: all expected months are present and schema is stable.


## Standardize Without Dropping Rows

Reason: the model needs consistent names and numeric types, but source fidelity matters. We keep original fields, create normalized fields, and add flags for suppression or parsing issues.

In [4]:
def clean_text(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip()


ma = pd.DataFrame(
    {
        "report_month": pd.PeriodIndex(raw_ma["report_month"], freq="M").to_timestamp(),
        "report_month_label": raw_ma["report_month"],
        "state": clean_text(raw_ma["State"]).str.upper(),
        "county": clean_text(raw_ma["County"]),
        "plan_type": clean_text(raw_ma["PLAN TYPE"]),
        "ssa_code_raw": clean_text(raw_ma["SSA Code"]),
        "fips_code_raw": clean_text(raw_ma["FIPS Code"]),
        "enrolled_raw": clean_text(raw_ma["Enrolled"]),
        "source_zip": raw_ma["source_zip"],
        "source_csv": raw_ma["source_csv"],
        "source_row_number": raw_ma["source_row_number"],
    }
)

ma["ssa_code"] = pd.to_numeric(ma["ssa_code_raw"], errors="coerce").astype("Int64")
ma["fips_code"] = pd.to_numeric(ma["fips_code_raw"], errors="coerce").astype("Int64")
ma["enrolled"] = pd.to_numeric(ma["enrolled_raw"].str.replace(",", "", regex=False), errors="coerce")

ma["is_enrollment_suppressed"] = ma["enrolled_raw"].eq(".")
ma["is_enrollment_blank"] = ma["enrolled_raw"].eq("")
ma["is_enrollment_numeric"] = ma["enrolled"].notna()
ma["has_state"] = ma["state"].ne("")
ma["has_county"] = ma["county"].ne("")
ma["has_plan_type"] = ma["plan_type"].ne("")
ma["has_fips"] = ma["fips_code"].notna()
ma["state_county_key"] = ma["state"] + "|" + ma["county"] + "|" + ma["fips_code_raw"]

ma = ma.sort_values(["report_month", "state", "county", "plan_type", "source_row_number"]).reset_index(drop=True)

ma.to_parquet(PROCESSED_DIR / "ma_scp_long.parquet", index=False)
ma.head(1000).to_csv(REPORT_TABLE_DIR / "ma_scp_long_profile_sample.csv", index=False)

display(ma.head(10))
print(f"Rows after standardization: {len(ma):,}")
print("No rows were dropped during standardization.")

,report_month,report_month_label,state,county,plan_type,ssa_code_raw,fips_code_raw,enrolled_raw,source_zip,source_csv,source_row_number,ssa_code,fips_code,enrolled,is_enrollment_suppressed,is_enrollment_blank,is_enrollment_numeric,has_state,has_county,has_plan_type,has_fips,state_county_key
0,2024-01-01,2024-01,AK,Aleutians East,HMO/HMOPOS,02013,02013,.,ma-scp-2024-01.zip,SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv,417,2013,2013,NaN,True,False,False,True,True,True,True,AK|Aleutians East|02013
1,2024-01-01,2024-01,AK,Aleutians East,LI NET Sponsor,02013,02013,.,ma-scp-2024-01.zip,SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv,418,2013,2013,NaN,True,False,False,True,True,True,True,AK|Aleutians East|02013
2,2024-01-01,2024-01,AK,Aleutians East,Local PPO,02013,02013,.,ma-scp-2024-01.zip,SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv,419,2013,2013,NaN,True,False,False,True,True,True,True,AK|Aleutians East|02013
3,2024-01-01,2024-01,AK,Aleutians East,MSA,02013,02013,.,ma-scp-2024-01.zip,SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv,420,2013,2013,NaN,True,False,False,True,True,True,True,AK|Aleutians East|02013
4,2024-01-01,2024-01,AK,Aleutians West,HMO/HMOPOS,02016,02016,.,ma-scp-2024-01.zip,SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv,421,2016,2016,NaN,True,False,False,True,True,True,True,AK|Aleutians West|02016
5,2024-01-01,2024-01,AK,Aleutians West,LI NET Sponsor,02016,02016,.,ma-scp-2024-01.zip,SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv,422,2016,2016,NaN,True,False,False,True,True,True,True,AK|Aleutians West|02016
6,2024-01-01,2024-01,AK,Aleutians West,Local PPO,02016,02016,.,ma-scp-2024-01.zip,SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv,423,2016,2016,NaN,True,False,False,True,True,True,True,AK|Aleutians West|02016
7,2024-01-01,2024-01,AK,Aleutians West,MSA,02016,02016,.,ma-scp-2024-01.zip,SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv,424,2016,2016,NaN,True,False,False,True,True,True,True,AK|Aleutians West|02016
8,2024-01-01,2024-01,AK,Anchorage,HMO/HMOPOS,02020,02020,60,ma-scp-2024-01.zip,SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv,425,2020,2020,60.0,False,False,True,True,True,True,True,AK|Anchorage|02020
9,2024-01-01,2024-01,AK,Anchorage,LI NET Sponsor,02020,02020,27,ma-scp-2024-01.zip,SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv,426,2020,2020,27.0,False,False,True,True,True,True,True,AK|Anchorage|02020


Rows after standardization: 590,398
No rows were dropped during standardization.


## Data Quality Profile

Reason: quality checks tell us how much of the dataset can be used directly for numeric forecasting and where suppression affects interpretation. Rows with suppressed enrollment are retained, but numeric sums should be described as lower-bound observed enrollment.

In [5]:
quality_by_month = (
    ma.groupby("report_month_label", as_index=False)
    .agg(
        rows=("enrolled_raw", "size"),
        numeric_rows=("is_enrollment_numeric", "sum"),
        suppressed_rows=("is_enrollment_suppressed", "sum"),
        blank_enrollment_rows=("is_enrollment_blank", "sum"),
        missing_state_rows=("has_state", lambda s: int((~s).sum())),
        missing_county_rows=("has_county", lambda s: int((~s).sum())),
        missing_plan_type_rows=("has_plan_type", lambda s: int((~s).sum())),
        missing_fips_rows=("has_fips", lambda s: int((~s).sum())),
    )
)
quality_by_month["suppressed_row_share"] = quality_by_month["suppressed_rows"] / quality_by_month["rows"]
quality_by_month["numeric_row_share"] = quality_by_month["numeric_rows"] / quality_by_month["rows"]

duplicate_keys = ["report_month", "state", "county", "fips_code_raw", "plan_type"]
dupe_mask = ma.duplicated(duplicate_keys, keep=False)
duplicate_profile = ma.loc[dupe_mask, duplicate_keys + ["enrolled_raw", "source_zip", "source_row_number"]].copy()

quality_by_month.to_csv(REPORT_TABLE_DIR / "ma_scp_quality_by_month.csv", index=False)
duplicate_profile.to_csv(REPORT_TABLE_DIR / "ma_scp_duplicate_key_rows.csv", index=False)

display(quality_by_month)
print(f"Duplicate key rows: {len(duplicate_profile):,}")
display(duplicate_profile.head(20))

,report_month_label,rows,numeric_rows,suppressed_rows,blank_enrollment_rows,missing_state_rows,missing_county_rows,missing_plan_type_rows,missing_fips_rows,suppressed_row_share,numeric_row_share
0,2024-01,20668,10512,10156,0,0,0,0,0,0.491388,0.508612
1,2024-02,20602,10651,9951,0,0,0,0,0,0.483011,0.516989
2,2024-03,20548,10550,9998,0,0,0,0,0,0.486568,0.513432
3,2024-04,20549,10617,9932,0,0,0,0,0,0.483333,0.516667
4,2024-05,20514,11168,9346,0,0,0,0,0,0.455591,0.544409
5,2024-06,20478,10679,9799,0,0,0,0,0,0.478514,0.521486
6,2024-07,20482,10565,9917,0,0,0,0,0,0.484181,0.515819
7,2024-08,20480,10595,9885,0,0,0,0,0,0.482666,0.517334
8,2024-09,20491,10883,9608,0,0,0,0,0,0.468889,0.531111
9,2024-10,20443,10748,9695,0,0,0,0,0,0.474245,0.525755


Duplicate key rows: 340


,report_month,state,county,fips_code_raw,plan_type,enrolled_raw,source_zip,source_row_number
1308,2024-01-01,CA,Los Angeles,06037,HMO/HMOPOS,808571,ma-scp-2024-01.zip,1276
1309,2024-01-01,CA,Los Angeles,06037,HMO/HMOPOS,10785,ma-scp-2024-01.zip,1283
1310,2024-01-01,CA,Los Angeles,06037,LI NET Sponsor,706,ma-scp-2024-01.zip,1277
1311,2024-01-01,CA,Los Angeles,06037,LI NET Sponsor,24,ma-scp-2024-01.zip,1284
1312,2024-01-01,CA,Los Angeles,06037,Local PPO,71828,ma-scp-2024-01.zip,1278
1313,2024-01-01,CA,Los Angeles,06037,Local PPO,255,ma-scp-2024-01.zip,1285
1314,2024-01-01,CA,Los Angeles,06037,MSA,.,ma-scp-2024-01.zip,1279
1315,2024-01-01,CA,Los Angeles,06037,MSA,.,ma-scp-2024-01.zip,1286
1316,2024-01-01,CA,Los Angeles,06037,Medicare-Medicaid Plan HMO/HMOPOS,87,ma-scp-2024-01.zip,1280
1317,2024-01-01,CA,Los Angeles,06037,Medicare-Medicaid Plan HMO/HMOPOS,.,ma-scp-2024-01.zip,1287


## Dimension Inventory

Reason: this tells us the modeling granularity available in the CMS source: states, counties/FIPS, and plan types. It also helps identify whether later dashboards can support state, county, and plan-type filters.

In [6]:
dimension_summary = pd.DataFrame(
    [
        {"dimension": "months", "distinct_count": ma["report_month_label"].nunique()},
        {"dimension": "states", "distinct_count": ma["state"].nunique()},
        {"dimension": "state_county_fips", "distinct_count": ma["state_county_key"].nunique()},
        {"dimension": "plan_types", "distinct_count": ma["plan_type"].nunique()},
        {"dimension": "source_rows", "distinct_count": len(ma)},
    ]
)

plan_type_inventory = (
    ma.groupby("plan_type", as_index=False)
    .agg(
        rows=("plan_type", "size"),
        months_present=("report_month_label", "nunique"),
        numeric_rows=("is_enrollment_numeric", "sum"),
        suppressed_rows=("is_enrollment_suppressed", "sum"),
        observed_enrollment=("enrolled", "sum"),
    )
    .sort_values("observed_enrollment", ascending=False)
)

state_inventory = (
    ma.groupby("state", as_index=False)
    .agg(
        rows=("state", "size"),
        counties=("state_county_key", "nunique"),
        plan_types=("plan_type", "nunique"),
        observed_enrollment=("enrolled", "sum"),
    )
    .sort_values("observed_enrollment", ascending=False)
)

dimension_summary.to_csv(REPORT_TABLE_DIR / "ma_scp_dimension_summary.csv", index=False)
plan_type_inventory.to_csv(REPORT_TABLE_DIR / "ma_scp_plan_type_inventory.csv", index=False)
state_inventory.to_csv(REPORT_TABLE_DIR / "ma_scp_state_inventory.csv", index=False)

display(dimension_summary)
display(plan_type_inventory)
display(state_inventory.head(15))

,dimension,distinct_count
0,months,29
1,states,56
2,state_county_fips,3268
3,plan_types,10
4,source_rows,590398


,plan_type,rows,months_present,numeric_rows,suppressed_rows,observed_enrollment
2,HMO/HMOPOS,94471,29,85496,8975,563029115.0
4,Local PPO,94490,29,91227,3263,425378620.0
9,Regional PPO,65934,29,54334,11600,7776530.0
6,Medicare-Medicaid Plan HMO/HMOPOS,28971,24,7975,20996,6047503.0
0,1876 Cost,14691,29,6363,8328,4911924.0
7,National PACE,24335,29,10568,13767,1945899.0
1,HCPP - 1833 Cost,42761,29,9605,33156,1358356.0
8,PFFS,41571,29,16554,25017,989106.0
3,LI NET Sponsor,92171,29,15826,76345,797177.0
5,MSA,91003,29,1088,89915,32963.0


,state,rows,counties,plan_types,observed_enrollment
5,CA,10702,58,10,104420323.0
10,FL,14332,67,10,85783494.0
47,TX,46160,254,10,76126393.0
37,NY,12835,62,10,60397171.0
41,PA,15314,67,10,46534751.0
38,OH,18872,88,10,41989289.0
24,MI,17046,83,10,41314052.0
30,NC,19906,100,10,37032087.0
11,GA,28213,159,10,31655622.0
16,IL,22128,102,10,30656470.0


## National Monthly Trend

Reason: the forecasting notebook needs a clean time series. This national trend is the simplest stable target and acts as the benchmark before state or plan-type forecasting.

In [7]:
monthly = (
    ma.groupby(["report_month", "report_month_label"], as_index=False)
    .agg(
        observed_enrollment=("enrolled", "sum"),
        source_rows=("enrolled_raw", "size"),
        numeric_rows=("is_enrollment_numeric", "sum"),
        suppressed_rows=("is_enrollment_suppressed", "sum"),
        states=("state", "nunique"),
        counties=("state_county_key", "nunique"),
        plan_types=("plan_type", "nunique"),
    )
    .sort_values("report_month")
)
monthly["mom_change"] = monthly["observed_enrollment"].diff()
monthly["mom_growth_pct"] = monthly["observed_enrollment"].pct_change()
monthly["indexed_to_first_month"] = monthly["observed_enrollment"] / monthly["observed_enrollment"].iloc[0] * 100
monthly["suppressed_row_share"] = monthly["suppressed_rows"] / monthly["source_rows"]

monthly.to_parquet(PROCESSED_DIR / "ma_scp_monthly_national.parquet", index=False)
monthly.to_csv(REPORT_TABLE_DIR / "ma_scp_monthly_national.csv", index=False)

display(monthly)
fig = px.line(monthly, x="report_month", y="observed_enrollment", markers=True, title="Observed MA Enrollment Trend - National")
fig.update_layout(yaxis_title="Observed enrollment", xaxis_title="Report month")
fig.show()

,report_month,report_month_label,observed_enrollment,source_rows,numeric_rows,suppressed_rows,states,counties,plan_types,mom_change,mom_growth_pct,indexed_to_first_month,suppressed_row_share
0,2024-01-01,2024-01,33476930.0,20668,10512,10156,56,3268,10,NaN,NaN,100.000000,0.491388
1,2024-02-01,2024-02,33668720.0,20602,10651,9951,56,3268,10,191790.0,0.005729,100.572902,0.483011
2,2024-03-01,2024-03,33805563.0,20548,10550,9998,56,3268,10,136843.0,0.004064,100.981670,0.486568
3,2024-04-01,2024-04,33899765.0,20549,10617,9932,56,3268,10,94202.0,0.002787,101.263064,0.483333
4,2024-05-01,2024-05,34053017.0,20514,11168,9346,56,3268,10,153252.0,0.004521,101.720848,0.455591
5,2024-06-01,2024-06,34103353.0,20478,10679,9799,56,3268,10,50336.0,0.001478,101.871208,0.478514
6,2024-07-01,2024-07,34165602.0,20482,10565,9917,56,3268,10,62249.0,0.001825,102.057154,0.484181
7,2024-08-01,2024-08,34254978.0,20480,10595,9885,56,3268,10,89376.0,0.002616,102.324132,0.482666
8,2024-09-01,2024-09,34340926.0,20491,10883,9608,56,3268,10,85948.0,0.002509,102.580870,0.468889
9,2024-10-01,2024-10,34419506.0,20443,10748,9695,56,3268,10,78580.0,0.002288,102.815599,0.474245


## Plan-Type Mix And Growth

Reason: plan type changes can explain aggregate movement. For RCM opportunity work, HMO/HMOPOS, Local PPO, Regional PPO, and similar categories may indicate different contracting or prior-auth workflows.

In [8]:
plan_monthly = (
    ma.groupby(["report_month", "report_month_label", "plan_type"], as_index=False)
    .agg(
        observed_enrollment=("enrolled", "sum"),
        numeric_rows=("is_enrollment_numeric", "sum"),
        suppressed_rows=("is_enrollment_suppressed", "sum"),
        counties=("state_county_key", "nunique"),
    )
    .sort_values(["plan_type", "report_month"])
)
plan_monthly["plan_mom_change"] = plan_monthly.groupby("plan_type")["observed_enrollment"].diff()
plan_monthly["plan_mom_growth_pct"] = plan_monthly.groupby("plan_type")["observed_enrollment"].pct_change()

month_totals = monthly[["report_month", "observed_enrollment"]].rename(columns={"observed_enrollment": "national_observed_enrollment"})
plan_monthly = plan_monthly.merge(month_totals, on="report_month", how="left")
plan_monthly["national_share"] = plan_monthly["observed_enrollment"] / plan_monthly["national_observed_enrollment"]

first_last_plan = (
    plan_monthly.sort_values("report_month")
    .groupby("plan_type")
    .agg(
        first_month=("report_month_label", "first"),
        last_month=("report_month_label", "last"),
        first_enrollment=("observed_enrollment", "first"),
        last_enrollment=("observed_enrollment", "last"),
        avg_share=("national_share", "mean"),
    )
    .reset_index()
)
first_last_plan["absolute_growth"] = first_last_plan["last_enrollment"] - first_last_plan["first_enrollment"]
first_last_plan["growth_pct"] = np.where(first_last_plan["first_enrollment"] > 0, first_last_plan["absolute_growth"] / first_last_plan["first_enrollment"], np.nan)
first_last_plan = first_last_plan.sort_values("last_enrollment", ascending=False)

plan_monthly.to_parquet(PROCESSED_DIR / "ma_scp_plan_monthly.parquet", index=False)
plan_monthly.to_csv(REPORT_TABLE_DIR / "ma_scp_plan_monthly.csv", index=False)
first_last_plan.to_csv(REPORT_TABLE_DIR / "ma_scp_plan_type_growth.csv", index=False)

display(first_last_plan)
top_plans = first_last_plan.head(8)["plan_type"].tolist()
fig = px.line(
    plan_monthly[plan_monthly["plan_type"].isin(top_plans)],
    x="report_month",
    y="observed_enrollment",
    color="plan_type",
    markers=True,
    title="Observed Enrollment by Major Plan Type",
)
fig.update_layout(yaxis_title="Observed enrollment", xaxis_title="Report month")
fig.show()

,plan_type,first_month,last_month,first_enrollment,last_enrollment,avg_share,absolute_growth,growth_pct
2,HMO/HMOPOS,2024-01,2026-05,18466930.0,20786875.0,0.556052,2319945.0,0.125627
4,Local PPO,2024-01,2026-05,14002520.0,14760496.0,0.420279,757976.0,0.054131
6,Medicare-Medicaid Plan HMO/HMOPOS,2024-01,2025-12,291288.0,218172.0,0.007273,-73116.0,-0.251009
0,1876 Cost,2024-01,2026-05,164180.0,196079.0,0.004852,31899.0,0.194293
9,Regional PPO,2024-01,2026-05,392516.0,153424.0,0.007736,-239092.0,-0.609127
7,National PACE,2024-01,2026-05,59276.0,76683.0,0.001920,17407.0,0.293660
1,HCPP - 1833 Cost,2024-01,2026-05,49157.0,44764.0,0.001343,-4393.0,-0.089367
8,PFFS,2024-01,2026-05,30699.0,33959.0,0.000976,3260.0,0.106192
3,LI NET Sponsor,2024-01,2026-05,19436.0,28741.0,0.000790,9305.0,0.478751
5,MSA,2024-01,2026-05,928.0,2260.0,0.000032,1332.0,1.435345


## State-Level Opportunity View

Reason: the dashboard and sales story need geography. This section identifies current scale, absolute growth, percentage growth, and volatility by state.

In [9]:
state_monthly = (
    ma.groupby(["report_month", "report_month_label", "state"], as_index=False)
    .agg(
        observed_enrollment=("enrolled", "sum"),
        source_rows=("enrolled_raw", "size"),
        numeric_rows=("is_enrollment_numeric", "sum"),
        suppressed_rows=("is_enrollment_suppressed", "sum"),
        counties=("state_county_key", "nunique"),
        plan_types=("plan_type", "nunique"),
    )
    .sort_values(["state", "report_month"])
)
state_monthly["mom_change"] = state_monthly.groupby("state")["observed_enrollment"].diff()
state_monthly["mom_growth_pct"] = state_monthly.groupby("state")["observed_enrollment"].pct_change()

state_growth = (
    state_monthly.groupby("state")
    .agg(
        first_month=("report_month_label", "first"),
        last_month=("report_month_label", "last"),
        first_enrollment=("observed_enrollment", "first"),
        last_enrollment=("observed_enrollment", "last"),
        avg_enrollment=("observed_enrollment", "mean"),
        min_enrollment=("observed_enrollment", "min"),
        max_enrollment=("observed_enrollment", "max"),
        avg_mom_growth_pct=("mom_growth_pct", "mean"),
        volatility_mom_growth_pct=("mom_growth_pct", "std"),
        counties=("counties", "max"),
    )
    .reset_index()
)
state_growth["absolute_growth"] = state_growth["last_enrollment"] - state_growth["first_enrollment"]
state_growth["growth_pct"] = np.where(state_growth["first_enrollment"] > 0, state_growth["absolute_growth"] / state_growth["first_enrollment"], np.nan)
state_growth["opportunity_rank"] = state_growth["absolute_growth"].rank(ascending=False, method="dense").astype(int)
state_growth = state_growth.sort_values(["absolute_growth", "last_enrollment"], ascending=False)

state_monthly.to_parquet(PROCESSED_DIR / "ma_scp_state_monthly.parquet", index=False)
state_monthly.to_csv(REPORT_TABLE_DIR / "ma_scp_state_monthly.csv", index=False)
state_growth.to_csv(REPORT_TABLE_DIR / "ma_scp_state_growth.csv", index=False)

display(state_growth.head(20))
fig = px.bar(state_growth.head(15), x="state", y="absolute_growth", title="Top States by Absolute Observed Enrollment Growth")
fig.update_layout(yaxis_title="Growth from first to last month", xaxis_title="State")
fig.show()

state_growth_for_plot = state_growth.copy()
state_growth_for_plot["abs_growth_for_marker"] = state_growth_for_plot["absolute_growth"].abs()

fig = px.scatter(
    state_growth_for_plot,
    x="last_enrollment",
    y="growth_pct",
    size="abs_growth_for_marker",
    hover_name="state",
    title="State Opportunity: Current Scale vs Growth Rate",
)
fig.update_layout(xaxis_title="Latest observed enrollment", yaxis_title="Growth rate from first to last month")
fig.show()

,state,first_month,last_month,first_enrollment,last_enrollment,avg_enrollment,min_enrollment,max_enrollment,avg_mom_growth_pct,volatility_mom_growth_pct,counties,absolute_growth,growth_pct,opportunity_rank
5,CA,2024-01,2026-05,3440093.0,3718525.0,3.600701e+06,3440093.0,3718525.0,0.002790,0.003548,58,278432.0,0.080937,1
47,TX,2024-01,2026-05,2510805.0,2748087.0,2.625048e+06,2510805.0,2748087.0,0.003233,0.002550,254,237282.0,0.094504,2
10,FL,2024-01,2026-05,2853027.0,3058222.0,2.958052e+06,2853027.0,3058222.0,0.002486,0.002403,67,205195.0,0.071922,3
37,NY,2024-01,2026-05,1985636.0,2169002.0,2.082661e+06,1976532.0,2169002.0,0.003264,0.014590,62,183366.0,0.092346,4
30,NC,2024-01,2026-05,1205103.0,1344776.0,1.276969e+06,1205103.0,1344776.0,0.003930,0.003469,100,139673.0,0.115901,5
41,PA,2024-01,2026-05,1541778.0,1657943.0,1.604647e+06,1541778.0,1657943.0,0.002610,0.004980,67,116165.0,0.075345,6
24,MI,2024-01,2026-05,1362242.0,1477625.0,1.424622e+06,1362242.0,1477625.0,0.002916,0.003982,83,115383.0,0.084701,7
38,OH,2024-01,2026-05,1396356.0,1487881.0,1.447907e+06,1396356.0,1487881.0,0.002277,0.003712,88,91525.0,0.065546,8
11,GA,2024-01,2026-05,1051421.0,1138849.0,1.091573e+06,1051421.0,1138849.0,0.002893,0.008662,159,87428.0,0.083152,9
52,WA,2024-01,2026-05,712478.0,787257.0,7.600052e+05,712478.0,787257.0,0.003729,0.017934,39,74779.0,0.104956,10


## County-Level Growth

Reason: county-level growth powers a regional opportunity map. We still keep suppressed rows, but county totals use observed numeric enrollment only.

In [10]:
county_monthly = (
    ma.groupby(["report_month", "report_month_label", "state", "county", "fips_code_raw", "state_county_key"], as_index=False)
    .agg(
        observed_enrollment=("enrolled", "sum"),
        source_rows=("enrolled_raw", "size"),
        numeric_rows=("is_enrollment_numeric", "sum"),
        suppressed_rows=("is_enrollment_suppressed", "sum"),
        plan_types=("plan_type", "nunique"),
    )
    .sort_values(["state_county_key", "report_month"])
)
county_monthly["mom_change"] = county_monthly.groupby("state_county_key")["observed_enrollment"].diff()
county_monthly["mom_growth_pct"] = county_monthly.groupby("state_county_key")["observed_enrollment"].pct_change()

county_growth = (
    county_monthly.groupby(["state_county_key", "state", "county", "fips_code_raw"], as_index=False)
    .agg(
        first_month=("report_month_label", "first"),
        last_month=("report_month_label", "last"),
        months_present=("report_month_label", "nunique"),
        first_enrollment=("observed_enrollment", "first"),
        last_enrollment=("observed_enrollment", "last"),
        avg_enrollment=("observed_enrollment", "mean"),
        total_suppressed_rows=("suppressed_rows", "sum"),
        avg_mom_growth_pct=("mom_growth_pct", "mean"),
        volatility_mom_growth_pct=("mom_growth_pct", "std"),
    )
)
county_growth["absolute_growth"] = county_growth["last_enrollment"] - county_growth["first_enrollment"]
county_growth["growth_pct"] = np.where(county_growth["first_enrollment"] > 0, county_growth["absolute_growth"] / county_growth["first_enrollment"], np.nan)
county_growth = county_growth.sort_values(["absolute_growth", "last_enrollment"], ascending=False)

county_monthly.to_parquet(PROCESSED_DIR / "ma_scp_county_monthly.parquet", index=False)
county_growth.to_csv(REPORT_TABLE_DIR / "ma_scp_county_growth.csv", index=False)

display(county_growth.head(25))
display(county_growth.sort_values("growth_pct", ascending=False).head(25))

,state_county_key,state,county,fips_code_raw,first_month,last_month,months_present,first_enrollment,last_enrollment,avg_enrollment,total_suppressed_rows,avg_mom_growth_pct,volatility_mom_growth_pct,absolute_growth,growth_pct
223,CA|Los Angeles|06037,CA,Los Angeles,06037,2024-01,2026-05,29,895944.0,977969.0,940978.206897,160,0.003136,0.002471,82025.0,0.091551
2745,TX|Harris|48201,TX,Harris,48201,2024-01,2026-05,29,374114.0,406879.0,389716.137931,65,0.003006,0.002376,32765.0,0.087580
234,CA|Orange|06059,CA,Orange,06059,2024-01,2026-05,29,310005.0,337909.0,326459.827586,104,0.003089,0.003540,27904.0,0.090011
2063,NY|Queens|36081,NY,Queens,36081,2024-01,2026-05,29,230574.0,258072.0,247094.517241,51,0.004043,0.004867,27498.0,0.119259
197,AZ|Maricopa|04013,AZ,Maricopa,04013,2024-01,2026-05,29,396118.0,421756.0,408236.586207,73,0.002247,0.003025,25638.0,0.064723
237,CA|Riverside|06065,CA,Riverside,06065,2024-01,2026-05,29,262387.0,286564.0,276733.827586,116,0.003158,0.003321,24177.0,0.092143
2046,NY|Kings|36047,NY,Kings,36047,2024-01,2026-05,29,215097.0,238569.0,230039.310345,57,0.003714,0.004095,23472.0,0.109123
750,IL|Cook|17031,IL,Cook,17031,2024-01,2026-05,29,378191.0,400583.0,389701.586207,66,0.002067,0.004622,22392.0,0.059208
241,CA|San Diego|06073,CA,San Diego,06073,2024-01,2026-05,29,313096.0,335079.0,324228.827586,115,0.002428,0.001937,21983.0,0.070212
240,CA|San Bernardino|06071,CA,San Bernardino,06071,2024-01,2026-05,29,216114.0,236469.0,227644.137931,100,0.003223,0.002575,20355.0,0.094186


,state_county_key,state,county,fips_code_raw,first_month,last_month,months_present,first_enrollment,last_enrollment,avg_enrollment,total_suppressed_rows,avg_mom_growth_pct,volatility_mom_growth_pct,absolute_growth,growth_pct
709,ID|Custer|16037,ID,Custer,16037,2024-01,2026-05,29,20.0,226.0,125.448276,75,0.255263,1.273362,206.0,10.300000
731,ID|Teton|16081,ID,Teton,16081,2024-01,2026-05,29,65.0,272.0,175.482759,75,0.089549,0.411557,207.0,3.184615
720,ID|Lemhi|16059,ID,Lemhi,16059,2024-01,2026-05,29,69.0,244.0,181.551724,73,0.068527,0.297731,175.0,2.536232
708,ID|Clearwater|16035,ID,Clearwater,16035,2024-01,2026-05,29,128.0,446.0,277.896552,75,0.062294,0.249370,318.0,2.484375
695,ID|Benewah|16009,ID,Benewah,16009,2024-01,2026-05,29,184.0,543.0,323.172414,63,0.049870,0.183958,359.0,1.951087
721,ID|Lewis|16061,ID,Lewis,16061,2024-01,2026-05,29,56.0,162.0,98.862069,92,0.059422,0.282100,106.0,1.892857
2916,UT|San Juan|49037,UT,San Juan,49037,2024-01,2026-05,29,152.0,428.0,289.275862,75,0.045925,0.156646,276.0,1.815789
2898,UT|Beaver|49001,UT,Beaver,49001,2024-01,2026-05,29,191.0,535.0,357.793103,75,0.047300,0.179245,344.0,1.801047
2910,UT|Kane|49025,UT,Kane,49025,2024-01,2026-05,29,217.0,578.0,388.965517,96,0.042377,0.143353,361.0,1.663594
1931,NE|Sioux|31165,NE,Sioux,31165,2024-01,2026-05,29,21.0,53.0,36.413793,94,0.037879,0.104226,32.0,1.523810


## Outlier And Volatility Checks

Reason: forecasting models can overreact to one-off jumps. This section identifies the largest national, state, and county month-over-month shifts for review before model training.

In [11]:
national_outliers = monthly.copy()
national_outliers["abs_mom_change"] = national_outliers["mom_change"].abs()

state_outliers = state_monthly.dropna(subset=["mom_change"]).copy()
state_outliers["abs_mom_change"] = state_outliers["mom_change"].abs()
state_outliers = state_outliers.sort_values("abs_mom_change", ascending=False)

county_outliers = county_monthly.dropna(subset=["mom_change"]).copy()
county_outliers["abs_mom_change"] = county_outliers["mom_change"].abs()
county_outliers = county_outliers.sort_values("abs_mom_change", ascending=False)

state_outliers.head(100).to_csv(REPORT_TABLE_DIR / "ma_scp_state_mom_outliers.csv", index=False)
county_outliers.head(250).to_csv(REPORT_TABLE_DIR / "ma_scp_county_mom_outliers.csv", index=False)

display(national_outliers[["report_month_label", "observed_enrollment", "mom_change", "mom_growth_pct"]])
display(state_outliers.head(20))
display(county_outliers.head(20))

,report_month_label,observed_enrollment,mom_change,mom_growth_pct
0,2024-01,33476930.0,NaN,NaN
1,2024-02,33668720.0,191790.0,0.005729
2,2024-03,33805563.0,136843.0,0.004064
3,2024-04,33899765.0,94202.0,0.002787
4,2024-05,34053017.0,153252.0,0.004521
5,2024-06,34103353.0,50336.0,0.001478
6,2024-07,34165602.0,62249.0,0.001825
7,2024-08,34254978.0,89376.0,0.002616
8,2024-09,34340926.0,85948.0,0.002509
9,2024-10,34419506.0,78580.0,0.002288


,report_month,report_month_label,state,observed_enrollment,source_rows,numeric_rows,suppressed_rows,counties,plan_types,mom_change,mom_growth_pct,abs_mom_change
709,2025-01-01,2025-01,NY,1976532.0,439,290,149,62,10,-104145.0,-0.050053,104145.0
1369,2026-01-01,2026-01,MN,646419.0,531,231,300,87,9,-73511.0,-0.102109,73511.0
1381,2026-01-01,2026-01,NY,2080640.0,427,224,203,62,9,-67150.0,-0.031265,67150.0
1437,2026-02-01,2026-02,NY,2144180.0,428,228,200,62,9,63540.0,0.030539,63540.0
724,2025-01-01,2025-01,WA,728774.0,209,105,104,39,10,-53837.0,-0.068792,53837.0
765,2025-02-01,2025-02,NY,2029745.0,441,287,154,62,10,53213.0,0.026922,53213.0
733,2025-02-01,2025-02,CA,3592432.0,365,178,187,58,10,36360.0,0.010225,36360.0
775,2025-02-01,2025-02,TX,2618488.0,1595,821,774,254,10,34089.0,0.013190,34089.0
612,2024-11-01,2024-11,WA,782141.0,224,99,125,39,10,32872.0,0.043872,32872.0
989,2025-06-01,2025-06,NY,2113884.0,444,290,154,62,10,32849.0,0.015785,32849.0


,report_month,report_month_label,state,county,fips_code_raw,state_county_key,observed_enrollment,source_rows,numeric_rows,suppressed_rows,plan_types,mom_change,mom_growth_pct,abs_mom_change
79813,2026-01-01,2026-01,MN,Hennepin,27053,MN|Hennepin|27053,123587.0,9,4,5,9,-16608.0,-0.118464,16608.0
41266,2025-01-01,2025-01,NY,Monroe,36055,NY|Monroe|36055,116428.0,9,6,3,9,-15632.0,-0.118370,15632.0
80482,2026-01-01,2026-01,NY,Monroe,36055,NY|Monroe|36055,123213.0,9,5,4,9,-11929.0,-0.088270,11929.0
80469,2026-01-01,2026-01,NY,Erie,36029,NY|Erie|36029,144449.0,7,5,2,7,-10471.0,-0.067590,10471.0
42311,2025-01-01,2025-01,WA,King,53033,WA|King|53033,188350.0,9,5,4,9,-10243.0,-0.051578,10243.0
83737,2026-02-01,2026-02,NY,Erie,36029,NY|Erie|36029,154462.0,7,5,2,7,10013.0,0.069319,10013.0
41253,2025-01-01,2025-01,NY,Erie,36029,NY|Erie|36029,141219.0,9,7,2,9,-9312.0,-0.061861,9312.0
6759,2024-03-01,2024-03,CA,Los Angeles,06037,CA|Los Angeles|06037,909608.0,15,10,5,9,9066.0,0.010067,9066.0
6363,2024-02-01,2024-02,WA,King,53033,WA|King|53033,188095.0,8,5,3,8,8977.0,0.050118,8977.0
83750,2026-02-01,2026-02,NY,Monroe,36055,NY|Monroe|36055,131979.0,9,5,4,9,8766.0,0.071145,8766.0


## Forecasting Readiness Summary

Reason: this translates EDA into modeling decisions for the next notebook. A good forecast notebook should not guess the target; it should inherit a documented target and caveats from EDA.

In [12]:
readiness = {
    "source_months": int(monthly["report_month_label"].nunique()),
    "source_rows_preserved": int(len(ma)),
    "national_first_month": monthly["report_month_label"].iloc[0],
    "national_last_month": monthly["report_month_label"].iloc[-1],
    "national_first_observed_enrollment": float(monthly["observed_enrollment"].iloc[0]),
    "national_last_observed_enrollment": float(monthly["observed_enrollment"].iloc[-1]),
    "national_absolute_growth": float(monthly["observed_enrollment"].iloc[-1] - monthly["observed_enrollment"].iloc[0]),
    "national_growth_pct": float(monthly["observed_enrollment"].iloc[-1] / monthly["observed_enrollment"].iloc[0] - 1),
    "avg_suppressed_row_share": float(monthly["suppressed_row_share"].mean()),
    "state_count": int(ma["state"].nunique()),
    "county_key_count": int(ma["state_county_key"].nunique()),
    "plan_type_count": int(ma["plan_type"].nunique()),
    "recommended_first_forecast_target": "monthly national observed_enrollment",
    "recommended_second_forecast_target": "state_monthly observed_enrollment for top-growth states",
    "main_caveat": "Rows with CMS suppressed enrollment are retained; numeric totals use observed numeric enrollment and should be described as observed/lower-bound enrollment.",
}

readiness_df = pd.DataFrame([readiness])
readiness_df.to_csv(REPORT_TABLE_DIR / "ma_scp_forecasting_readiness.csv", index=False)
display(readiness_df.T.rename(columns={0: "value"}))

print("EDA outputs written to data/processed and reports/tables.")

,value
source_months,29
source_rows_preserved,590398
national_first_month,2024-01
national_last_month,2026-05
national_first_observed_enrollment,33476930.0
national_last_observed_enrollment,36083281.0
national_absolute_growth,2606351.0
national_growth_pct,0.077855
avg_suppressed_row_share,0.493198
state_count,56


EDA outputs written to data/processed and reports/tables.
